# EquiDrug - Colab Training Notebook (self-contained)

Task 3.2. **Fully self-contained**: every source file is embedded directly in
this notebook via `%%writefile` cells and written to disk here - no git clone,
no zip upload. Running this notebook top-to-bottom on a free Colab GPU:

1. Writes the full `src/` tree (config, data pipeline, all three models, training loop) to disk.
2. Installs the pinned free/open-source dependencies from an embedded `requirements.txt`.
3. Trains Vanilla GNN, EGNN, and SE(3)-Transformer.
4. Uploads the three best-weight checkpoints + CSV logs to a **public** Hugging Face Hub repo (free).

**Free-cost guardrails (CLAUDE.md):** Colab free tier only, QM9 subset (not the full ~130k), public HF repo only. If anything here ever prompts for billing, STOP.

## 0. Confirm we have a free GPU
Runtime -> Change runtime type -> T4 GPU (free tier) before running this cell.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU detected - set Runtime > Change runtime type > T4 GPU (free), then re-run."

## 1. Create the project directory structure

In [ ]:
import os

REPO_DIR = "/content/equidrug"
for sub in ["src/data", "src/models", "src/train", "weights", "data"]:
    os.makedirs(os.path.join(REPO_DIR, sub), exist_ok=True)
%cd {REPO_DIR}
print("Project root:", os.getcwd())

## 2. Write every source file to disk
Each cell below embeds one file verbatim (identical to the local repo) and writes it via `%%writefile`.

### `requirements.txt`

In [ ]:
%%writefile requirements.txt
# EquiDrug — pinned free/open-source dependencies
# All packages below are free and open source (no paid APIs, no credit card).
# Python 3.10+ required.

# --- Core ML framework ---
torch==2.2.2                 # PyTorch, BSD-style license
torch-geometric==2.5.3       # PyTorch Geometric (graph NN), MIT

# --- Geometry / equivariance ---
e3nn==0.5.1                  # SE(3)-equivariant building blocks, MIT

# --- Molecule parsing / 3D embedding ---
rdkit==2023.9.6              # RDKit (maintained PyPI package), BSD

# --- Numerics / data ---
numpy==1.26.4                # pinned <2.0 for torch/pyg compatibility, BSD
pandas==2.2.2                # BSD

# --- Backend API ---
fastapi==0.111.0             # MIT
uvicorn==0.30.1              # ASGI server, BSD


### `verify_env.py`

In [ ]:
%%writefile verify_env.py
"""Environment verification for EquiDrug.

Imports every library in the free stack and prints its version, confirming
the dependencies in requirements.txt install cleanly. Run this after
`pip install -r requirements.txt` (locally on CPU or on Colab).

Usage:
    python verify_env.py

Exit code is 0 if every library imports, 1 if any are missing.
"""

from __future__ import annotations

import importlib


# (pip package name, import module name, attribute holding the version)
LIBRARIES = [
    ("torch", "torch", "__version__"),
    ("torch-geometric", "torch_geometric", "__version__"),
    ("e3nn", "e3nn", "__version__"),
    ("rdkit", "rdkit", "__version__"),
    ("numpy", "numpy", "__version__"),
    ("pandas", "pandas", "__version__"),
    ("fastapi", "fastapi", "__version__"),
    ("uvicorn", "uvicorn", "__version__"),
]


def main() -> int:
    print("EquiDrug environment check")
    print("=" * 40)

    missing: list[str] = []
    for pip_name, module_name, version_attr in LIBRARIES:
        try:
            module = importlib.import_module(module_name)
            version = getattr(module, version_attr, "unknown")
            print(f"  [ok]   {pip_name:18} {version}")
        except ImportError as exc:
            print(f"  [MISS] {pip_name:18} -> {exc}")
            missing.append(pip_name)

    print("=" * 40)

    # Report compute device via our central config (also exercises the import).
    try:
        from src.train.config import get_device

        print(f"Device detected: {get_device()}")
    except Exception as exc:  # noqa: BLE001 - report any config import issue
        print(f"Device detect:   could not import config ({exc})")

    if missing:
        print(f"\nFAILED: {len(missing)} package(s) missing: {', '.join(missing)}")
        print("Run: pip install -r requirements.txt")
        return 1

    print("\nAll libraries imported successfully. Free stack is ready.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `src/train/config.py`

In [ ]:
%%writefile src/train/config.py
"""Global configuration for EquiDrug.

Central place for the random seed, device auto-detection, and all
hyperparameters. Import from here everywhere so runs stay reproducible
and consistent across the three models (Vanilla GNN, EGNN, SE(3)-Transformer).

This module has NO heavy dependencies beyond torch/numpy so it is safe to
import early (e.g. from verify_env.py or any training script).
"""

from __future__ import annotations

import os
import random
from dataclasses import dataclass, field

import numpy as np

try:
    import torch

    _TORCH_AVAILABLE = True
except ImportError:  # keep config importable even before torch is installed
    torch = None  # type: ignore
    _TORCH_AVAILABLE = False


# --------------------------------------------------------------------------- #
# Reproducibility
# --------------------------------------------------------------------------- #
SEED: int = 42


def set_seed(seed: int = SEED, deterministic: bool = True) -> None:
    """Seed every RNG we touch so results are reproducible.

    Seeds Python's `random`, NumPy, and (if available) PyTorch on both CPU
    and CUDA. Call this once at the start of any script that trains or
    evaluates a model.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

    if _TORCH_AVAILABLE:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        if deterministic:
            # Favor reproducibility over the last few % of speed.
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False


def get_device() -> "str":
    """Auto-detect compute device: CUDA if present, else CPU.

    Local development/smoke-testing returns 'cpu'; Colab's free GPU
    returns 'cuda'. Returns a string so the module imports even when torch
    is not installed yet.
    """
    if _TORCH_AVAILABLE and torch.cuda.is_available():
        return "cuda"
    return "cpu"


DEVICE: str = get_device()


# --------------------------------------------------------------------------- #
# Hyperparameters
# --------------------------------------------------------------------------- #
@dataclass(frozen=True)
class DataConfig:
    """Dataset loading / splitting (see Phase 1)."""

    subset_size: int = 50_000      # QM9 subset per CLAUDE.md free-cost guardrail
    train_frac: float = 0.8
    val_frac: float = 0.1
    test_frac: float = 0.1         # train + val + test must sum to 1.0
    # Prediction targets (QM9 column names handled in qm9_loader.py).
    targets: tuple[str, ...] = ("homo_lumo_gap", "dipole_moment")


@dataclass(frozen=True)
class ModelConfig:
    """Shared architecture sizes; kept identical across models for fairness."""

    hidden_dim: int = 128
    num_layers: int = 4
    out_dim: int = 2               # must match len(DataConfig.targets)
    dropout: float = 0.0


@dataclass(frozen=True)
class TrainConfig:
    """Optimization / training loop settings."""

    epochs: int = 100
    batch_size: int = 96
    lr: float = 1e-3
    weight_decay: float = 1e-5
    grad_clip: float = 10.0
    # Mixed precision on Colab GPU (free-cost guardrail); ignored on CPU.
    use_amp: bool = True
    num_workers: int = 2
    # Paths (relative to repo root).
    weights_dir: str = "weights"
    log_dir: str = "weights"       # CSV metric logs live alongside weights


@dataclass(frozen=True)
class RotationConfig:
    """Rotation stress test / ESS settings (see Phase 4)."""

    num_rotations: int = 24        # every 15 degrees, 0..345


@dataclass(frozen=True)
class Config:
    """Top-level config bundling all sections plus global settings."""

    seed: int = SEED
    device: str = field(default_factory=get_device)
    data: DataConfig = field(default_factory=DataConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    train: TrainConfig = field(default_factory=TrainConfig)
    rotation: RotationConfig = field(default_factory=RotationConfig)


# Convenient ready-made instance for imports: `from src.train.config import CONFIG`
CONFIG = Config()


if __name__ == "__main__":
    set_seed()
    print(f"Seed:   {CONFIG.seed}")
    print(f"Device: {CONFIG.device}")
    print(f"Data:   {CONFIG.data}")
    print(f"Model:  {CONFIG.model}")
    print(f"Train:  {CONFIG.train}")
    print(f"Rotate: {CONFIG.rotation}")
    # Sanity: splits sum to 1.0 and out_dim matches number of targets.
    d = CONFIG.data
    assert abs(d.train_frac + d.val_frac + d.test_frac - 1.0) < 1e-9, "splits must sum to 1"
    assert CONFIG.model.out_dim == len(d.targets), "out_dim must match number of targets"
    print("\nConfig self-check passed.")


### `src/data/qm9_loader.py`

In [ ]:
%%writefile src/data/qm9_loader.py
"""QM9 data loading + reproducible subsetting and splitting.

Task 1.1: load QM9 via PyTorch Geometric, subset to ~50k molecules
(free-cost guardrail — the full ~130k dataset is unnecessary and slower),
and split into train/val/test using a fixed seed so every run sees the
exact same molecules in each split.

QM9 auto-downloads into ``data/`` (gitignored) the first time this runs.
Each sample is a PyG ``Data`` object with 3D atom coordinates in ``pos``
and 19 regression targets in ``y`` (target *selection* and units are
handled later, in task 1.2 — not here).

Run directly to do a self-check:

    python -m src.data.qm9_loader
"""

from __future__ import annotations

import os
import sys
from dataclasses import dataclass

import torch
from torch_geometric.datasets import QM9

# Make `from src.train.config import CONFIG` work whether this file is run as a
# module (`python -m src.data.qm9_loader`) or imported from elsewhere.
_REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

from src.train.config import CONFIG, set_seed  # noqa: E402

# QM9 downloads/processes into this directory (relative to repo root).
DEFAULT_ROOT = os.path.join(_REPO_ROOT, "data", "QM9")


@dataclass(frozen=True)
class QM9Splits:
    """Container for the three split datasets plus the indices behind them.

    ``train``/``val``/``test`` are PyG datasets (subsets of QM9). The index
    tensors are kept so the exact split is auditable and reproducible.
    """

    train: QM9
    val: QM9
    test: QM9
    train_idx: torch.Tensor
    val_idx: torch.Tensor
    test_idx: torch.Tensor

    def sizes(self) -> dict[str, int]:
        return {
            "train": len(self.train),
            "val": len(self.val),
            "test": len(self.test),
        }


def load_qm9(root: str = DEFAULT_ROOT) -> QM9:
    """Load (and on first run, download + process) the full QM9 dataset."""
    return QM9(root=root)


def _subset_and_split_indices(
    n_total: int,
    subset_size: int,
    train_frac: float,
    val_frac: float,
    test_frac: float,
    seed: int,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Deterministically pick a subset of size ``subset_size`` then split it.

    A single seeded permutation drives everything: the first ``subset_size``
    indices form the working subset, which is then carved into train/val/test.
    Same seed -> identical splits on every machine and every run.
    """
    assert abs(train_frac + val_frac + test_frac - 1.0) < 1e-9, "fractions must sum to 1.0"

    # Clamp in case the dataset is smaller than the requested subset.
    subset_size = min(subset_size, n_total)

    generator = torch.Generator().manual_seed(seed)
    perm = torch.randperm(n_total, generator=generator)
    subset = perm[:subset_size]

    n_train = int(round(train_frac * subset_size))
    n_val = int(round(val_frac * subset_size))
    # Test gets the remainder so the three always sum to subset_size exactly.
    train_idx = subset[:n_train]
    val_idx = subset[n_train : n_train + n_val]
    test_idx = subset[n_train + n_val :]
    return train_idx, val_idx, test_idx


def get_qm9_splits(
    root: str = DEFAULT_ROOT,
    subset_size: int | None = None,
    train_frac: float | None = None,
    val_frac: float | None = None,
    test_frac: float | None = None,
    seed: int | None = None,
) -> QM9Splits:
    """Load QM9 and return reproducible train/val/test split datasets.

    Defaults come from ``CONFIG.data`` / ``CONFIG.seed`` so the whole project
    shares one definition of the splits. Pass overrides for quick experiments.
    """
    dc = CONFIG.data
    subset_size = dc.subset_size if subset_size is None else subset_size
    train_frac = dc.train_frac if train_frac is None else train_frac
    val_frac = dc.val_frac if val_frac is None else val_frac
    test_frac = dc.test_frac if test_frac is None else test_frac
    seed = CONFIG.seed if seed is None else seed

    set_seed(seed)  # seed global RNGs too, for good measure

    dataset = load_qm9(root)
    train_idx, val_idx, test_idx = _subset_and_split_indices(
        n_total=len(dataset),
        subset_size=subset_size,
        train_frac=train_frac,
        val_frac=val_frac,
        test_frac=test_frac,
        seed=seed,
    )

    # PyG datasets support indexing with a LongTensor -> a new subset dataset.
    return QM9Splits(
        train=dataset[train_idx],
        val=dataset[val_idx],
        test=dataset[test_idx],
        train_idx=train_idx,
        val_idx=val_idx,
        test_idx=test_idx,
    )


# --------------------------------------------------------------------------- #
# Task 1.2 — Prediction targets, units, and normalization
# --------------------------------------------------------------------------- #
# We predict exactly two QM9 properties (see CLAUDE.md / CONFIG.data.targets):
#
#   * HOMO-LUMO gap  (energy gap between highest occupied and lowest
#     unoccupied molecular orbital)
#   * Dipole moment  (magnitude of the molecular dipole)
#
# QM9's per-molecule label tensor ``data.y`` has shape [1, 19]. PyTorch
# Geometric stores these 19 columns in a FIXED order and, during processing,
# converts the orbital energies from Hartree to electron-volts (eV) for us.
# The two columns we use (verified empirically against this dataset):
#
#   col 0  ->  dipole moment (mu)        unit: Debye (D),  always >= 0
#   col 4  ->  HOMO-LUMO gap (delta-eps) unit: eV,         equals LUMO - HOMO
#
# Mapping from human-readable target name -> column index in ``data.y``.
QM9_TARGET_COLUMN: dict[str, int] = {
    "dipole_moment": 0,    # mu,  Debye
    "homo_lumo_gap": 4,    # Delta epsilon = LUMO - HOMO,  eV
}

# Physical units, documented here so predictions can always be reported in the
# correct unit (and so the frontend can label axes).
QM9_TARGET_UNIT: dict[str, str] = {
    "homo_lumo_gap": "eV",
    "dipole_moment": "Debye",
}


def get_target_columns(targets: tuple[str, ...] | None = None) -> list[int]:
    """Return the ``data.y`` column indices for our targets, in config order.

    Order matches ``CONFIG.data.targets`` (and therefore the model's output
    dimension), so model output channel ``i`` always corresponds to
    ``targets[i]``.
    """
    targets = CONFIG.data.targets if targets is None else targets
    return [QM9_TARGET_COLUMN[name] for name in targets]


def select_targets(
    y: torch.Tensor, targets: tuple[str, ...] | None = None
) -> torch.Tensor:
    """Slice the two columns we predict out of a full QM9 ``y`` tensor.

    Accepts either a single label ``[1, 19]`` or a batched ``[B, 19]`` tensor
    and returns ``[*, len(targets)]`` in config target order (raw physical
    units — not yet normalized).
    """
    cols = get_target_columns(targets)
    return y[..., cols]


@dataclass(frozen=True)
class TargetNormalizer:
    """Standardize targets to zero mean / unit variance for stable training.

    WHY normalize: the gap (~0.7-17 eV) and the dipole (~0-30 D) live on very
    different scales and units. Regressing raw values lets the larger-magnitude
    target dominate the loss. We standardize each target independently:

        z = (y - mean) / std          # used as the training label
        y = z * std + mean            # invert to recover physical units

    WHY train-split statistics only: ``mean``/``std`` are computed from the
    TRAIN split exclusively, never val/test, so no information leaks from the
    evaluation data into preprocessing. The same fixed stats are then applied
    to val/test and at inference time.

    Predictions come out of the model in normalized space and MUST be passed
    through :meth:`denormalize` before being reported in eV / Debye.
    """

    mean: torch.Tensor  # shape [num_targets], physical units
    std: torch.Tensor   # shape [num_targets], physical units
    targets: tuple[str, ...]

    @classmethod
    def from_dataset(
        cls, train_dataset, targets: tuple[str, ...] | None = None
    ) -> "TargetNormalizer":
        """Fit normalization statistics on a (train) dataset's targets."""
        targets = CONFIG.data.targets if targets is None else targets
        cols = get_target_columns(targets)
        # Stack the selected target columns across all molecules: [N, num_targets].
        ys = torch.stack([data.y[0, cols] for data in train_dataset])
        mean = ys.mean(dim=0)
        std = ys.std(dim=0)
        std = torch.where(std > 0, std, torch.ones_like(std))  # guard /0
        return cls(mean=mean, std=std, targets=tuple(targets))

    def normalize(self, y: torch.Tensor) -> torch.Tensor:
        """Physical units -> standardized. ``y`` may be raw [*,19] or [*,T]."""
        if y.shape[-1] != len(self.targets):
            y = select_targets(y, self.targets)
        return (y - self.mean) / self.std

    def denormalize(self, z: torch.Tensor) -> torch.Tensor:
        """Standardized -> physical units (eV / Debye). Inverse of normalize."""
        return z * self.std + self.mean

    def units(self) -> list[str]:
        return [QM9_TARGET_UNIT[name] for name in self.targets]


if __name__ == "__main__":
    splits = get_qm9_splits()
    sizes = splits.sizes()
    total = sum(sizes.values())

    print("QM9 loaded and split (seed =", CONFIG.seed, ")")
    print(f"  subset size : {total}")
    print(f"  train       : {sizes['train']}")
    print(f"  val         : {sizes['val']}")
    print(f"  test        : {sizes['test']}")

    # Reproducibility + disjointness checks.
    all_idx = torch.cat([splits.train_idx, splits.val_idx, splits.test_idx])
    assert all_idx.unique().numel() == all_idx.numel(), "splits overlap!"
    assert total == min(CONFIG.data.subset_size, total) or total <= CONFIG.data.subset_size

    # Re-run the split and confirm it is identical (deterministic).
    splits2 = get_qm9_splits()
    assert torch.equal(splits.train_idx, splits2.train_idx), "split not reproducible!"

    sample = splits.train[0]
    print("\nSample molecule from train split:")
    print(" ", sample)
    print("  has 3D coords (pos):", hasattr(sample, "pos") and sample.pos.shape[1] == 3)

    # ---- Task 1.2: targets, units, normalization ---------------------------
    print("\nPrediction targets (config order):")
    for name in CONFIG.data.targets:
        print(f"  {name:14s} -> y[:, {QM9_TARGET_COLUMN[name]}]  ({QM9_TARGET_UNIT[name]})")

    # Sanity: column 4 is the gap (== LUMO - HOMO) and dipole is non-negative.
    y_full = sample.y  # [1, 19]
    gap = y_full[0, 4]
    homo, lumo = y_full[0, 2], y_full[0, 3]
    assert torch.isclose(gap, lumo - homo, atol=1e-2), "col 4 is not HOMO-LUMO gap"
    assert y_full[0, 0] >= 0, "dipole moment should be non-negative"

    sel = select_targets(y_full)  # [1, 2] in (gap, dipole) order
    assert sel.shape[-1] == len(CONFIG.data.targets) == CONFIG.model.out_dim

    # Fit normalizer on the TRAIN split only, then check the round-trip.
    norm = TargetNormalizer.from_dataset(splits.train)
    print("\nNormalization stats (fit on train split only):")
    for i, name in enumerate(CONFIG.data.targets):
        print(
            f"  {name:14s} mean={norm.mean[i]:8.4f}  std={norm.std[i]:8.4f}"
            f"  [{QM9_TARGET_UNIT[name]}]"
        )

    z = norm.normalize(y_full)
    back = norm.denormalize(z)
    assert torch.allclose(back, sel, atol=1e-4), "normalize/denormalize not invertible"

    # Standardized train targets should have ~0 mean / ~1 std by construction.
    ys_train = torch.stack([d.y[0, get_target_columns()] for d in splits.train])
    zs = norm.normalize(ys_train)
    assert zs.mean(0).abs().max() < 1e-3 and (zs.std(0) - 1).abs().max() < 1e-2
    print("  standardized train mean:", [round(v, 5) for v in zs.mean(0).tolist()])
    print("  standardized train std :", [round(v, 5) for v in zs.std(0).tolist()])

    print("\nqm9_loader self-check passed.")


### `src/data/molecule_utils.py`

In [ ]:
%%writefile src/data/molecule_utils.py
"""SMILES -> 3D molecular graph (RDKit) in PyTorch Geometric ``Data`` format.

Task 1.3: turn a user-supplied SMILES string into the same kind of graph the
models were trained on. We:

  1. Parse the SMILES with RDKit and add explicit hydrogens (QM9 molecules
     include every H as a real atom, so we must too).
  2. Generate a 3D conformer with ``EmbedMolecule`` (ETKDG, seeded for
     reproducibility).
  3. Relax the geometry with ``MMFFOptimizeMolecule`` (UFF fallback).
  4. Extract atoms, 3D coordinates, and bonds into a PyG ``Data`` object whose
     node/edge features EXACTLY match PyG's QM9 encoding, so a model trained on
     QM9 can run on this molecule unchanged.

Feature layout (matches ``torch_geometric.datasets.QM9``):
  * ``x``         [N, 11] : one-hot[H,C,N,O,F] (5) + atomic_number, aromatic,
                            sp, sp2, sp3, num_bonded_H  (6)
  * ``z``         [N]     : atomic numbers (int64)
  * ``pos``       [N, 3]  : 3D coordinates in angstroms (float)
  * ``edge_index``[2, E]  : bonds, both directions (undirected -> 2 edges each)
  * ``edge_attr`` [E, 4]  : one-hot bond type [single, double, triple, aromatic]
  * ``smiles``            : the input string (for reference / display)

There is no ``y`` — the property is unknown until a model predicts it.
"""

from __future__ import annotations

import os
import sys

import torch
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.rdchem import BondType, HybridizationType
from torch_geometric.data import Data

# Allow running both as a module and standalone (mirrors qm9_loader.py).
_REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

from src.train.config import CONFIG  # noqa: E402

# --- Encoding tables (identical to PyG's QM9 processing) -------------------- #
# Atom symbol -> index into the 5-way one-hot. QM9 only contains H,C,N,O,F.
ATOM_TYPES: dict[str, int] = {"H": 0, "C": 1, "N": 2, "O": 3, "F": 4}

# Bond type -> index into the 4-way one-hot.
BOND_TYPES: dict[BondType, int] = {
    BondType.SINGLE: 0,
    BondType.DOUBLE: 1,
    BondType.TRIPLE: 2,
    BondType.AROMATIC: 3,
}


class MoleculeEmbedError(ValueError):
    """Raised when a SMILES cannot be parsed or embedded into 3D."""


def _embed_3d(mol: Chem.Mol, seed: int) -> Chem.Mol:
    """Add explicit Hs, generate a seeded 3D conformer, and relax it.

    Returns the H-added molecule with exactly one optimized 3D conformer.
    Raises :class:`MoleculeEmbedError` if a conformer cannot be produced.
    """
    mol = Chem.AddHs(mol)

    params = AllChem.ETKDGv3()
    params.randomSeed = seed  # deterministic coordinates for a given SMILES
    if AllChem.EmbedMolecule(mol, params) != 0:
        # Retry with random starting coordinates for awkward geometries.
        params.useRandomCoords = True
        if AllChem.EmbedMolecule(mol, params) != 0:
            raise MoleculeEmbedError("RDKit could not embed a 3D conformer")

    # Geometry optimization: MMFF if parameters are available, else UFF.
    if AllChem.MMFFHasAllMoleculeParams(mol):
        AllChem.MMFFOptimizeMolecule(mol)
    else:
        AllChem.UFFOptimizeMolecule(mol)
    return mol


def _atom_features(mol: Chem.Mol) -> tuple[torch.Tensor, torch.Tensor]:
    """Build the [N, 11] node-feature matrix ``x`` and atomic numbers ``z``."""
    type_idx, atomic_number, aromatic = [], [], []
    sp, sp2, sp3, num_hs = [], [], [], []

    for atom in mol.GetAtoms():
        symbol = atom.GetSymbol()
        if symbol not in ATOM_TYPES:
            raise MoleculeEmbedError(
                f"atom '{symbol}' is outside QM9's element set {list(ATOM_TYPES)}"
            )
        type_idx.append(ATOM_TYPES[symbol])
        atomic_number.append(atom.GetAtomicNum())
        aromatic.append(1 if atom.GetIsAromatic() else 0)
        hyb = atom.GetHybridization()
        sp.append(1 if hyb == HybridizationType.SP else 0)
        sp2.append(1 if hyb == HybridizationType.SP2 else 0)
        sp3.append(1 if hyb == HybridizationType.SP3 else 0)
        # Count bonded hydrogens directly (Hs are explicit after AddHs).
        num_hs.append(sum(1 for nb in atom.GetNeighbors() if nb.GetAtomicNum() == 1))

    z = torch.tensor(atomic_number, dtype=torch.long)
    # 5-way one-hot of atom type.
    x1 = torch.nn.functional.one_hot(
        torch.tensor(type_idx), num_classes=len(ATOM_TYPES)
    ).to(torch.float)
    # 6 scalar/binary features, stacked to [N, 6].
    x2 = torch.tensor(
        [atomic_number, aromatic, sp, sp2, sp3, num_hs], dtype=torch.float
    ).t().contiguous()
    x = torch.cat([x1, x2], dim=-1)  # [N, 11]
    return x, z


def _bond_features(mol: Chem.Mol) -> tuple[torch.Tensor, torch.Tensor]:
    """Build ``edge_index`` [2, E] and one-hot ``edge_attr`` [E, 4].

    Each chemical bond becomes two directed edges (i->j and j->i) so the graph
    is undirected, matching QM9.
    """
    rows, cols, bond_idx = [], [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bt = bond.GetBondType()
        if bt not in BOND_TYPES:
            raise MoleculeEmbedError(f"unsupported bond type: {bt}")
        # Add both directions.
        rows += [i, j]
        cols += [j, i]
        bond_idx += [BOND_TYPES[bt], BOND_TYPES[bt]]

    edge_index = torch.tensor([rows, cols], dtype=torch.long)
    edge_attr = torch.nn.functional.one_hot(
        torch.tensor(bond_idx), num_classes=len(BOND_TYPES)
    ).to(torch.float)
    return edge_index, edge_attr


def smiles_to_data(smiles: str, seed: int | None = None) -> Data:
    """Convert a SMILES string into a QM9-compatible 3D graph ``Data`` object.

    Args:
        smiles: the molecule as a SMILES string (e.g. ``"CCO"`` for ethanol).
        seed:   random seed for 3D embedding; defaults to ``CONFIG.seed`` so the
                same SMILES always yields the same coordinates.

    Raises:
        MoleculeEmbedError: if the SMILES is invalid, contains non-QM9 elements,
            or cannot be embedded into 3D.
    """
    seed = CONFIG.seed if seed is None else seed

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise MoleculeEmbedError(f"invalid SMILES: {smiles!r}")

    mol = _embed_3d(mol, seed)

    x, z = _atom_features(mol)
    edge_index, edge_attr = _bond_features(mol)

    conf = mol.GetConformer()
    pos = torch.tensor(conf.GetPositions(), dtype=torch.float)  # [N, 3] angstrom

    return Data(
        x=x,
        z=z,
        pos=pos,
        edge_index=edge_index,
        edge_attr=edge_attr,
        smiles=Chem.MolToSmiles(mol),
    )


if __name__ == "__main__":
    # Smoke test on a few molecules (a fuller ethanol sanity check is task 1.4).
    examples = {
        "ethanol": "CCO",
        "benzene": "c1ccccc1",
        "acetonitrile": "CC#N",
    }
    for name, smi in examples.items():
        data = smiles_to_data(smi)
        n = data.num_nodes
        assert data.x.shape == (n, 11), "x must be [N, 11] (QM9-compatible)"
        assert data.pos.shape == (n, 3), "pos must be [N, 3]"
        assert data.z.shape == (n,)
        assert data.edge_index.shape[0] == 2
        assert data.edge_attr.shape == (data.edge_index.shape[1], 4)
        # Coordinates must be genuinely 3D (not collapsed onto a plane/line).
        spread = data.pos.max(dim=0).values - data.pos.min(dim=0).values
        assert (spread > 1e-3).sum() >= 2, "coordinates do not look 3D"
        print(f"{name:13s} {data.smiles:20s} atoms={n:3d} bonds={data.edge_index.shape[1]//2:3d}")

    # Element validation should reject atoms outside QM9's set (e.g. sulfur).
    try:
        smiles_to_data("CS")  # methanethiol -> contains S
        raise SystemExit("expected MoleculeEmbedError for sulfur-containing SMILES")
    except MoleculeEmbedError:
        pass

    print("\nmolecule_utils self-check passed.")


### `src/models/vanilla_gnn.py`

In [ ]:
%%writefile src/models/vanilla_gnn.py
"""Vanilla GNN baseline: a GIN that uses node features only.

Task 2.1 — the *intentionally weak* model. This is the "bad example" the whole
project is built to expose: a graph neural network that knows WHICH atoms are
bonded to which (topology) and WHAT each atom is (node features), but is
completely **blind to 3D geometry**.

Why blind on purpose
--------------------
It reads only ``data.x`` (the [N, 11] QM9 atom features: element one-hot +
atomic number, aromaticity, hybridization, bonded-H count) and the bond graph
``data.edge_index``. It NEVER touches ``data.pos`` (the 3D coordinates). So it
cannot tell apart two conformers, and — crucially for our rotation stress test
(Phase 4) — rotating a molecule does not change ``x`` or ``edge_index`` at all,
so this model's prediction is trivially constant under rotation... *for a fixed
input graph*.

The interesting failure shows up the other way around: because it ignores
geometry entirely, it has no access to the very information (bond angles,
interatomic distances, 3D shape) that actually determines properties like the
dipole moment. The EGNN and SE(3)-Transformer do use ``pos`` equivariantly and
should therefore be both more accurate AND provably stable. This baseline is
the foil that makes that contrast measurable.

GIN (Graph Isomorphism Network, Xu et al. 2019) is chosen because it is a
strong, standard, purely topological message-passing model — so when it loses,
it loses on the merits of ignoring geometry, not because it is a toy.

Architecture (sizes shared via ``CONFIG.model`` for a fair comparison):
    x [N, 11]
      -> Linear encoder            -> [N, hidden]
      -> num_layers x GINConv      -> [N, hidden]   (topology only)
      -> global_add_pool           -> [B, hidden]   (graph-level readout)
      -> MLP head                  -> [B, out_dim]  (the predicted targets)

Run a standalone smoke test (CPU, no dataset download needed):

    python -m src.models.vanilla_gnn
"""

from __future__ import annotations

import os
import sys

import torch
from torch import nn
from torch_geometric.nn import GINConv, global_add_pool

# Make `from src.train.config import CONFIG` work whether run as a module
# (`python -m src.models.vanilla_gnn`) or imported elsewhere (mirrors the
# data-pipeline modules).
_REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

from src.train.config import CONFIG  # noqa: E402

# QM9 / molecule_utils node-feature width (see src/data/molecule_utils.py).
NODE_FEATURE_DIM: int = 11


def _gin_mlp(in_dim: int, hidden_dim: int) -> nn.Sequential:
    """The 2-layer MLP that GIN applies after summing neighbor messages.

    GINConv aggregates neighbors by SUM and then transforms with this MLP; a
    2-layer MLP gives the conv enough capacity to be an injective (and thus
    maximally discriminative) aggregator, which is GIN's whole point.
    """
    return nn.Sequential(
        nn.Linear(in_dim, hidden_dim),
        nn.BatchNorm1d(hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, hidden_dim),
        nn.ReLU(),
    )


class VanillaGNN(nn.Module):
    """Topology-only GIN regressor (the geometry-blind baseline).

    Args:
        in_dim:     node-feature width (defaults to QM9's 11).
        hidden_dim: width of every hidden layer (default ``CONFIG.model.hidden_dim``).
        num_layers: number of GINConv message-passing layers.
        out_dim:    number of regression targets (default ``CONFIG.model.out_dim``).
        dropout:    dropout applied in the prediction head.

    forward(x, edge_index, batch) -> [B, out_dim]
        Deliberately takes NO ``pos`` argument — there is no way to feed this
        model 3D coordinates, which is the point.
    """

    def __init__(
        self,
        in_dim: int = NODE_FEATURE_DIM,
        hidden_dim: int | None = None,
        num_layers: int | None = None,
        out_dim: int | None = None,
        dropout: float | None = None,
    ) -> None:
        super().__init__()
        mc = CONFIG.model
        hidden_dim = mc.hidden_dim if hidden_dim is None else hidden_dim
        num_layers = mc.num_layers if num_layers is None else num_layers
        out_dim = mc.out_dim if out_dim is None else out_dim
        dropout = mc.dropout if dropout is None else dropout

        # Project raw node features up to the hidden width before message passing.
        self.encoder = nn.Linear(in_dim, hidden_dim)

        # Stack of GIN convolutions. Each one mixes a node with its bonded
        # neighbors (edge_index) only — no coordinates ever enter here.
        self.convs = nn.ModuleList(
            GINConv(_gin_mlp(hidden_dim, hidden_dim), train_eps=True)
            for _ in range(num_layers)
        )

        # Graph-level prediction head applied after pooling node embeddings.
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        batch: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """Predict graph-level targets from node features + bond topology.

        Args:
            x:          [N, in_dim] node features.
            edge_index: [2, E] bond graph (both directions).
            batch:      [N] graph-id per node for batched pooling; if ``None``,
                        all nodes are treated as one graph.

        Returns:
            [B, out_dim] predictions (normalized space — denormalize for units).
        """
        if batch is None:
            batch = x.new_zeros(x.size(0), dtype=torch.long)

        h = self.encoder(x)
        for conv in self.convs:
            # Residual connection keeps gradients healthy across the stack.
            h = h + conv(h, edge_index)

        # SUM pooling matches GIN's injective-aggregation philosophy at the
        # graph level: collapse [N, hidden] node embeddings -> [B, hidden].
        h = global_add_pool(h, batch)
        return self.head(h)


def build_vanilla_gnn(**overrides) -> VanillaGNN:
    """Convenience factory mirroring ``CONFIG`` defaults; pass kwargs to override."""
    return VanillaGNN(**overrides)


if __name__ == "__main__":
    # CPU smoke test: build the model and push two tiny fake molecules through
    # it. No dataset download required — we just need the shapes to line up and
    # to confirm the model genuinely ignores coordinates.
    torch.manual_seed(CONFIG.seed)

    model = build_vanilla_gnn()
    n_params = sum(p.numel() for p in model.parameters())
    print("VanillaGNN (GIN baseline):")
    print(f"  hidden_dim={CONFIG.model.hidden_dim}  num_layers={CONFIG.model.num_layers}"
          f"  out_dim={CONFIG.model.out_dim}  params={n_params:,}")

    # Fake batch: molecule A has 3 atoms, molecule B has 2 atoms (5 nodes total).
    x = torch.randn(5, NODE_FEATURE_DIM)
    edge_index = torch.tensor(
        [[0, 1, 1, 2, 3, 4],
         [1, 0, 2, 1, 4, 3]], dtype=torch.long
    )
    batch = torch.tensor([0, 0, 0, 1, 1], dtype=torch.long)

    model.eval()
    with torch.no_grad():
        out = model(x, edge_index, batch)
    assert out.shape == (2, CONFIG.model.out_dim), f"bad output shape: {out.shape}"
    print(f"  forward OK: input 5 nodes / 2 graphs -> output {tuple(out.shape)}")

    # Single-graph path (batch=None) should also work and give [1, out_dim].
    with torch.no_grad():
        out1 = model(x[:3], edge_index[:, :4])
    assert out1.shape == (1, CONFIG.model.out_dim)

    # KEY PROPERTY: the model is geometry-blind. Its forward signature has no
    # `pos` argument, so 3D coordinates (and any rotation of them) cannot affect
    # the output. We demonstrate that the output depends ONLY on (x, edge_index)
    # by re-running with identical inputs and getting identical results.
    with torch.no_grad():
        out_again = model(x, edge_index, batch)
    assert torch.allclose(out, out_again), "model should be deterministic on fixed inputs"
    print("  geometry-blind: forward() takes no `pos`; output is a function of (x, edge_index) only")

    print("\nvanilla_gnn self-check passed.")


### `src/models/egnn.py`

In [ ]:
%%writefile src/models/egnn.py
"""EGNN — E(n) Equivariant Graph Neural Network (Satorras, Hoogeboom & Welling, 2021).

Task 2.2 — the CORE of the project: a graph network that DOES use 3D geometry,
and does so in a provably E(3)-equivariant way. This is the model that should
stay rock-steady under rotation (ESS ≈ 1.0) where the Vanilla GIN baseline,
having no notion of geometry at all, cannot benefit from coordinates.

Paper: "E(n) Equivariant Graph Neural Networks", ICML 2021.
        https://arxiv.org/abs/2102.09844   (~150 lines of real math; no e3nn.)

----------------------------------------------------------------------------
The four EGNN equations (one message-passing layer, for every edge i<-j)
----------------------------------------------------------------------------
Let h_i be the (invariant) feature vector of node i and x_i its 3D position.

  (1) edge message      m_ij = phi_e( h_i, h_j, ||x_i - x_j||^2, a_ij )
  (2) coordinate update x_i' = x_i + C * sum_j (x_i - x_j) * phi_x(m_ij)
  (3) aggregate         m_i  = sum_j  m_ij
  (4) node update       h_i' = phi_h( h_i, m_i )

phi_e, phi_x, phi_h are plain MLPs. a_ij are optional edge attributes (we feed
the QM9 bond-type one-hot). C is a normalizer (we use 1/(#neighbors)).

THE KEY IDEA, in one line:
  positions enter the network ONLY through the squared distance ||x_i - x_j||^2.

Distance is invariant to rotation, translation, AND reflection — i.e. to the
whole Euclidean group E(3). So every message m_ij (and therefore every node
feature h_i) is an E(3)-INVARIANT scalar. The coordinate update (2) moves atoms
along the relative vectors (x_i - x_j) scaled by those invariant weights, which
makes the positions transform EQUIVARIANTLY. See the long note on
``EGNNLayer._coord_update`` below for the full why-it-works argument.

What this means for us:
  * We read out the graph-level property from the final INVARIANT features h.
    => the predicted HOMO-LUMO gap / dipole magnitude does NOT change when the
       molecule is rotated, translated, or reflected. That is exactly the
       robustness the Equivariance Stability Score (Phase 4) is designed to
       reward.

Architecture:
    (h0 from x[N,11]) , pos[N,3]
      -> encoder Linear: x -> h [N, hidden]
      -> num_layers x EGNNLayer  (updates h invariantly, pos equivariantly)
      -> global_add_pool over h  -> [B, hidden]
      -> MLP head                -> [B, out_dim]

Run a standalone smoke + equivariance test (CPU, no dataset needed):

    python -m src.models.egnn
"""

from __future__ import annotations

import os
import sys

import torch
from torch import nn
from torch_geometric.nn import global_add_pool
from torch_geometric.utils import degree

# Make `from src.train.config import CONFIG` work whether run as a module
# (`python -m src.models.egnn`) or imported elsewhere (mirrors the other modules).
_REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

from src.train.config import CONFIG  # noqa: E402

# QM9 / molecule_utils widths (see src/data/molecule_utils.py).
NODE_FEATURE_DIM: int = 11   # data.x
EDGE_FEATURE_DIM: int = 4    # data.edge_attr (bond-type one-hot)


class EGNNLayer(nn.Module):
    """One E(n)-equivariant message-passing layer (eqs. 1-4 above).

    Updates node features ``h`` invariantly and node positions ``pos``
    equivariantly. Stacking several of these is the whole EGNN.
    """

    def __init__(
        self,
        hidden_dim: int,
        edge_dim: int = EDGE_FEATURE_DIM,
        update_coords: bool = True,
    ) -> None:
        super().__init__()
        self.update_coords = update_coords

        # phi_e : edge MLP. Input = [h_i, h_j, dist^2, a_ij].
        #   The "+1" is the single scalar squared-distance feature — this is the
        #   ONLY way geometry enters, and being a distance it is E(3)-invariant.
        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * hidden_dim + 1 + edge_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
        )

        # phi_x : maps an edge message -> a single SCALAR weight for the
        # coordinate update. Final layer has no bias and is zero-initialized so
        # the model starts as (near) the identity on coordinates — a standard
        # EGNN stabilization trick.
        self.coord_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1, bias=False),
        )
        nn.init.zeros_(self.coord_mlp[-1].weight)

        # phi_h : node MLP. Input = [h_i, aggregated_message_i].
        self.node_mlp = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def _coord_update(
        self,
        pos: torch.Tensor,
        rel: torch.Tensor,
        row: torch.Tensor,
        edge_weight: torch.Tensor,
        num_nodes: int,
    ) -> torch.Tensor:
        """Equivariant coordinate update — eq. (2). Read the note carefully.

        ``rel = pos[row] - pos[col]`` is the relative vector along each edge,
        ``edge_weight = phi_x(m_ij)`` is a learned INVARIANT scalar per edge.

            x_i' = x_i + C * sum_{j in N(i)} (x_i - x_j) * phi_x(m_ij)

        ------------------------------------------------------------------
        WHY THIS IS E(3)-EQUIVARIANT (rotation + translation + reflection)
        ------------------------------------------------------------------
        Apply any rigid transform to every atom:   x  ->  R x + t,
        where R is an orthogonal matrix (rotation OR reflection, R^T R = I)
        and t is a translation.

          * Relative vectors lose the translation and rotate with R:
                (x_i + t-stuff) ...  (x_i - x_j) -> R x_i - R x_j = R (x_i - x_j).
            The +t cancels in the difference, so translation has NO effect here.

          * The weights phi_x(m_ij) depend on positions only via ||x_i - x_j||^2,
            and ||R(x_i - x_j)|| = ||x_i - x_j|| because R is orthogonal. So the
            messages — and hence every weight — are UNCHANGED by the transform.
            They are invariant scalars.

          * Therefore each term transforms as
                (x_i - x_j) * w_ij  ->  R (x_i - x_j) * w_ij,
            and summing linearly:
                x_i' = x_i + C * sum_j (x_i - x_j) w_ij
                     ->  R x_i + t + C * sum_j R (x_i - x_j) w_ij
                     =  R ( x_i + C * sum_j (x_i - x_j) w_ij ) + t
                     =  R x_i' + t.

        The updated coordinates transform by exactly the SAME (R, t) as the
        input. That is the definition of equivariance. No spherical harmonics,
        no e3nn — equivariance falls out of (a) using only invariant distances
        as geometric input and (b) only ever moving atoms along relative
        vectors. The node features h, seen only through invariant messages,
        stay invariant throughout — so a readout from h is E(3)-INVARIANT,
        which is what we want for predicting scalar molecular properties.
        ------------------------------------------------------------------
        """
        # Per-edge contribution: relative vector scaled by its invariant weight.
        trans = rel * edge_weight                       # [E, 3]
        # Sum contributions into the source node i (index_add over `row`).
        agg = pos.new_zeros((num_nodes, 3))
        agg.index_add_(0, row, trans)                   # sum_j (x_i - x_j) w_ij
        # C = 1 / deg(i): mean over neighbors keeps the update scale sane
        # regardless of how many bonds an atom has. (deg is invariant too.)
        deg = degree(row, num_nodes=num_nodes, dtype=pos.dtype).clamp(min=1).unsqueeze(-1)
        return pos + agg / deg

    def forward(
        self,
        h: torch.Tensor,
        pos: torch.Tensor,
        edge_index: torch.Tensor,
        edge_attr: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Run one layer. Returns updated (h, pos).

        Args:
            h:          [N, hidden] invariant node features.
            pos:        [N, 3] coordinates.
            edge_index: [2, E] (row = i = source, col = j = neighbor).
            edge_attr:  [E, edge_dim] bond-type one-hot (a_ij).
        """
        row, col = edge_index  # row -> i, col -> j
        num_nodes = h.size(0)

        # --- eq. (1): edge messages ---------------------------------------- #
        rel = pos[row] - pos[col]                       # [E, 3] relative vectors
        # Squared distance is the ONLY geometric input -> E(3)-invariant scalar.
        dist2 = (rel ** 2).sum(dim=-1, keepdim=True)    # [E, 1]
        edge_in = torch.cat([h[row], h[col], dist2, edge_attr], dim=-1)
        m_ij = self.edge_mlp(edge_in)                   # [E, hidden], invariant

        # --- eq. (2): equivariant coordinate update ------------------------ #
        if self.update_coords:
            edge_weight = self.coord_mlp(m_ij)          # [E, 1] invariant weight
            pos = self._coord_update(pos, rel, row, edge_weight, num_nodes)

        # --- eq. (3): aggregate messages into each node -------------------- #
        m_i = h.new_zeros((num_nodes, m_ij.size(-1)))
        m_i.index_add_(0, row, m_ij)                    # sum_j m_ij

        # --- eq. (4): node feature update (residual) ----------------------- #
        h = h + self.node_mlp(torch.cat([h, m_i], dim=-1))
        return h, pos


class EGNN(nn.Module):
    """EGNN regressor: invariant readout from E(3)-equivariant message passing.

    Args:
        in_dim:        node-feature width (defaults to QM9's 11).
        edge_dim:      edge-feature width (defaults to QM9's 4 bond types).
        hidden_dim:    hidden width (default ``CONFIG.model.hidden_dim``).
        num_layers:    number of EGNN layers (default ``CONFIG.model.num_layers``).
        out_dim:       number of regression targets (default ``CONFIG.model.out_dim``).
        dropout:       dropout in the prediction head.
        update_coords: if True (default) atoms move equivariantly between layers;
                       set False to keep coordinates fixed (pure distance model).

    forward(x, pos, edge_index, edge_attr, batch) -> [B, out_dim]
        The output is read from the final INVARIANT features, so it does not
        change under rotation / translation / reflection of ``pos``.
    """

    def __init__(
        self,
        in_dim: int = NODE_FEATURE_DIM,
        edge_dim: int = EDGE_FEATURE_DIM,
        hidden_dim: int | None = None,
        num_layers: int | None = None,
        out_dim: int | None = None,
        dropout: float | None = None,
        update_coords: bool = True,
    ) -> None:
        super().__init__()
        mc = CONFIG.model
        hidden_dim = mc.hidden_dim if hidden_dim is None else hidden_dim
        num_layers = mc.num_layers if num_layers is None else num_layers
        out_dim = mc.out_dim if out_dim is None else out_dim
        dropout = mc.dropout if dropout is None else dropout

        # Embed raw node features -> hidden width. This sees NO coordinates, so
        # the initial features are trivially invariant.
        self.encoder = nn.Linear(in_dim, hidden_dim)

        self.layers = nn.ModuleList(
            EGNNLayer(hidden_dim, edge_dim=edge_dim, update_coords=update_coords)
            for _ in range(num_layers)
        )

        # Graph-level head applied to pooled INVARIANT node features.
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(
        self,
        x: torch.Tensor,
        pos: torch.Tensor,
        edge_index: torch.Tensor,
        edge_attr: torch.Tensor,
        batch: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """Predict graph-level targets equivariantly.

        Args:
            x:          [N, in_dim] node features.
            pos:        [N, 3] coordinates (used ONLY via invariant distances).
            edge_index: [2, E] bond graph (both directions).
            edge_attr:  [E, edge_dim] bond-type one-hot.
            batch:      [N] graph-id per node; ``None`` -> single graph.

        Returns:
            [B, out_dim] predictions (normalized space — denormalize for units).
        """
        if batch is None:
            batch = x.new_zeros(x.size(0), dtype=torch.long)

        h = self.encoder(x)
        for layer in self.layers:
            h, pos = layer(h, pos, edge_index, edge_attr)

        # Read out from h only -> the prediction is E(3)-invariant by design.
        h = global_add_pool(h, batch)
        return self.head(h)


def build_egnn(**overrides) -> EGNN:
    """Convenience factory mirroring ``CONFIG`` defaults; pass kwargs to override."""
    return EGNN(**overrides)


def _random_rotation(seed: int = 0) -> torch.Tensor:
    """A random proper rotation matrix (det = +1) via QR, for the equiv test."""
    g = torch.Generator().manual_seed(seed)
    a = torch.randn(3, 3, generator=g)
    q, r = torch.linalg.qr(a)
    # Fix signs so the diagonal of R is positive -> q is a uniform rotation.
    q = q * torch.sign(torch.diagonal(r)).unsqueeze(0)
    if torch.det(q) < 0:        # ensure a rotation, not a reflection
        q[:, 0] = -q[:, 0]
    return q


if __name__ == "__main__":
    # CPU smoke + EQUIVARIANCE test. No dataset download needed: we build a tiny
    # fake batch and check (a) shapes line up and (b) the model is genuinely
    # E(3)-invariant in its readout — the property this whole module exists for.
    torch.manual_seed(CONFIG.seed)

    model = build_egnn()
    n_params = sum(p.numel() for p in model.parameters())
    print("EGNN (E(3)-equivariant, from scratch):")
    print(f"  hidden_dim={CONFIG.model.hidden_dim}  num_layers={CONFIG.model.num_layers}"
          f"  out_dim={CONFIG.model.out_dim}  params={n_params:,}")

    # Fake batch: molecule A (3 atoms), molecule B (2 atoms) -> 5 nodes.
    x = torch.randn(5, NODE_FEATURE_DIM)
    pos = torch.randn(5, 3)
    edge_index = torch.tensor(
        [[0, 1, 1, 2, 3, 4],
         [1, 0, 2, 1, 4, 3]], dtype=torch.long
    )
    edge_attr = torch.tensor(
        [[1, 0, 0, 0], [1, 0, 0, 0], [0, 1, 0, 0],
         [0, 1, 0, 0], [1, 0, 0, 0], [1, 0, 0, 0]], dtype=torch.float
    )
    batch = torch.tensor([0, 0, 0, 1, 1], dtype=torch.long)

    model.eval()
    with torch.no_grad():
        out = model(x, pos, edge_index, edge_attr, batch)
    assert out.shape == (2, CONFIG.model.out_dim), f"bad output shape: {out.shape}"
    print(f"  forward OK: 5 nodes / 2 graphs -> output {tuple(out.shape)}")

    # ---- E(3)-INVARIANCE of the prediction ------------------------------- #
    # Transform every coordinate by a random rotation R and a translation t.
    # Because positions enter only through invariant distances and the readout
    # is from invariant features, the output must be (numerically) unchanged.
    R = _random_rotation(seed=1)
    t = torch.tensor([3.0, -1.5, 0.7])
    pos_transformed = pos @ R.T + t            # x -> R x + t  (per row)

    with torch.no_grad():
        out_transformed = model(x, pos_transformed, edge_index, edge_attr, batch)

    max_diff = (out - out_transformed).abs().max().item()
    print(f"  E(3) invariance: max|f(x) - f(Rx + t)| = {max_diff:.2e}")
    assert max_diff < 1e-4, "prediction is NOT invariant under rotation+translation!"

    # Reflection (improper transform) should also leave the readout unchanged,
    # since distances are reflection-invariant too -> full E(3), not just SE(3).
    reflect = torch.diag(torch.tensor([-1.0, 1.0, 1.0]))
    with torch.no_grad():
        out_reflected = model(x, pos @ reflect.T, edge_index, edge_attr, batch)
    refl_diff = (out - out_reflected).abs().max().item()
    print(f"  reflection invariance: max diff = {refl_diff:.2e}")
    assert refl_diff < 1e-4, "prediction is NOT reflection-invariant!"

    print("\negnn self-check passed (forward shapes + E(3) invariance verified).")


### `src/models/se3_transformer.py`

In [ ]:
%%writefile src/models/se3_transformer.py
"""SE(3)-equivariant model built on the ``e3nn`` library (NOT from scratch).

Task 2.3 — the third model in the comparison. Where the EGNN earns its
equivariance from a single clever trick (only ever using invariant distances +
moving along relative vectors), this model earns it from the full machinery of
irreducible representations of SO(3): spherical harmonics of the edge vectors
combined through equivariant tensor products. That machinery is exactly what
``e3nn`` exists to provide, and CLAUDE.md is explicit: **use e3nn, do NOT
reinvent the math.** So this file is deliberately thin — it wires QM9 data into
e3nn's prebuilt gate-points message-passing network and reads out two invariant
scalars.

Why e3nn's gate-points network is SE(3)-equivariant (the short version)
----------------------------------------------------------------------
Every quantity in the network carries an ``Irreps`` type — a label saying how
it transforms under rotation (l=0 scalars are invariant, l=1 vectors rotate,
etc.). The convolution:

  1. takes each edge vector ``r_ij = pos_i - pos_j`` and expands its DIRECTION
     in spherical harmonics Y_l(r_ij / |r_ij|)  -> these transform as l-irreps,
  2. embeds the edge LENGTH |r_ij| (a rotation-invariant scalar) in a radial
     basis to produce the tensor-product weights,
  3. combines neighbor features with those harmonics via tensor products whose
     output irreps are chosen so the result transforms correctly,
  4. applies equivariant nonlinearities (gates) that never mix incompatible
     irreps.

Because length is invariant and direction is handled by spherical harmonics,
rotating the molecule rotates every internal feature *consistently with its
irrep*. We request an output of pure scalars (``2x0e``), so the prediction is
SE(3)-INVARIANT: rotating or translating the molecule cannot change it. (Edge
vectors are built as differences, so translation drops out — exactly like the
EGNN.) That invariance is what the rotation stress test / ESS rewards.

Note on the name: CLAUDE.md's stack table calls this the "SE(3)-Transformer".
The canonical e3nn realization of an SE(3)-equivariant point network is this
tensor-product gate-points convolution (Tensor-Field-Network lineage). We build
on it directly rather than hand-rolling attention, per the "do NOT implement
from scratch" instruction.

This model uses ``pos`` (3D geometry) directly; it builds its OWN radius graph
internally from coordinates, so — unlike the Vanilla GIN and EGNN — it does not
consume the precomputed bond ``edge_index`` / ``edge_attr``. We still accept
those arguments so all three models share one forward signature.

Run a standalone smoke + SE(3)-invariance test (CPU, no dataset needed):

    python -m src.models.se3_transformer
"""

from __future__ import annotations

import os
import sys

import torch
from torch import nn
from e3nn import o3
from e3nn.nn.models.v2106.gate_points_networks import SimpleNetwork

# Make `from src.train.config import CONFIG` work whether run as a module
# (`python -m src.models.se3_transformer`) or imported elsewhere.
_REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

from src.train.config import CONFIG  # noqa: E402

# QM9 / molecule_utils node-feature width (see src/data/molecule_utils.py).
# All 11 features are rotation-invariant scalars -> irreps "11x0e".
NODE_FEATURE_DIM: int = 11

# Geometry / normalization defaults tuned for small QM9 molecules. These are
# physical-ish hyperparameters, all in angstroms / counts, and are free to
# adjust later from a config without affecting equivariance.
DEFAULT_MAX_RADIUS: float = 5.0     # neighbor cutoff for the internal radius graph (Å)
DEFAULT_NUM_NEIGHBORS: float = 12.0  # avg neighbors within the cutoff (a normalizer)
DEFAULT_NUM_NODES: float = 18.0      # avg atoms per QM9 molecule (a normalizer)


class SE3Transformer(nn.Module):
    """SE(3)-equivariant regressor wrapping e3nn's gate-points network.

    Args:
        in_dim:        node-feature width (QM9 = 11 invariant scalars).
        out_dim:       number of regression targets (default ``CONFIG.model.out_dim``).
        max_radius:    cutoff (Å) for the internal radius graph over ``pos``.
        num_neighbors: average neighbor count (normalizer, not a hard limit).
        num_nodes:     average atoms per graph (pooling normalizer).
        mul:           multiplicity of each hidden irrep (≈ "hidden width").
        num_layers:    number of equivariant conv layers (default ``CONFIG.model.num_layers``).
        lmax:          max spherical-harmonic degree used (rotation order).

    forward(x, pos, edge_index, edge_attr, batch) -> [B, out_dim]
        Output irreps are pure scalars (``Nx0e``) so the prediction is
        SE(3)-invariant. ``edge_index`` / ``edge_attr`` are accepted for a
        common signature but unused — geometry comes from ``pos`` directly.
    """

    def __init__(
        self,
        in_dim: int = NODE_FEATURE_DIM,
        out_dim: int | None = None,
        max_radius: float = DEFAULT_MAX_RADIUS,
        num_neighbors: float = DEFAULT_NUM_NEIGHBORS,
        num_nodes: float = DEFAULT_NUM_NODES,
        mul: int = 32,
        num_layers: int | None = None,
        lmax: int = 2,
    ) -> None:
        super().__init__()
        mc = CONFIG.model
        out_dim = mc.out_dim if out_dim is None else out_dim
        num_layers = mc.num_layers if num_layers is None else num_layers

        # Irreps describe how each tensor transforms under rotation:
        #   input  = `in_dim` invariant scalars  -> "11x0e"
        #   output = `out_dim` invariant scalars -> "2x0e"  (SE(3)-invariant)
        self.irreps_in = o3.Irreps(f"{in_dim}x0e")
        self.irreps_out = o3.Irreps(f"{out_dim}x0e")

        # e3nn does ALL the equivariant heavy lifting: spherical harmonics of
        # edge directions, radial-basis edge-length embedding, equivariant
        # tensor-product convolutions, and gated nonlinearities.
        self.net = SimpleNetwork(
            irreps_in=self.irreps_in,
            irreps_out=self.irreps_out,
            max_radius=max_radius,
            num_neighbors=num_neighbors,
            num_nodes=num_nodes,
            mul=mul,
            layers=num_layers,
            lmax=lmax,
            pool_nodes=True,  # graph-level readout (sum over atoms, normalized)
        )

    def forward(
        self,
        x: torch.Tensor,
        pos: torch.Tensor,
        edge_index: torch.Tensor | None = None,
        edge_attr: torch.Tensor | None = None,
        batch: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """Predict graph-level targets SE(3)-equivariantly.

        Args:
            x:          [N, in_dim] invariant node features.
            pos:        [N, 3] coordinates (geometry source; radius graph built here).
            edge_index: accepted for a common signature; UNUSED.
            edge_attr:  accepted for a common signature; UNUSED.
            batch:      [N] graph-id per node; ``None`` -> single graph.

        Returns:
            [B, out_dim] predictions (normalized space — denormalize for units).
        """
        # e3nn's SimpleNetwork consumes a dict and builds the graph from `pos`.
        data = {"pos": pos, "x": x}
        if batch is not None:
            data["batch"] = batch
        out = self.net(data)
        # When pooling a single graph e3nn returns shape [out_dim]; make it
        # [1, out_dim] so the output is always [B, out_dim] like the other models.
        if out.dim() == 1:
            out = out.unsqueeze(0)
        return out


def build_se3_transformer(**overrides) -> SE3Transformer:
    """Convenience factory mirroring ``CONFIG`` defaults; pass kwargs to override."""
    return SE3Transformer(**overrides)


def _random_rotation(seed: int = 0) -> torch.Tensor:
    """A random proper rotation matrix (det = +1) via QR, for the equiv test."""
    g = torch.Generator().manual_seed(seed)
    a = torch.randn(3, 3, generator=g)
    q, r = torch.linalg.qr(a)
    q = q * torch.sign(torch.diagonal(r)).unsqueeze(0)
    if torch.det(q) < 0:        # ensure a rotation, not a reflection
        q[:, 0] = -q[:, 0]
    return q


if __name__ == "__main__":
    # CPU smoke + SE(3)-INVARIANCE test. No dataset download needed: build a
    # small fake molecule and check (a) shapes line up and (b) the prediction
    # is genuinely invariant to rotation + translation — the property this
    # whole module exists to provide.
    torch.manual_seed(CONFIG.seed)

    # Keep it small for a fast CPU test (e3nn tensor products are heavier than
    # plain GNN ops). Geometry is what matters here, not size.
    model = build_se3_transformer(mul=16, num_layers=2, lmax=2)
    n_params = sum(p.numel() for p in model.parameters())
    print("SE3Transformer (e3nn gate-points, SE(3)-equivariant):")
    print(f"  irreps_in={model.irreps_in}  irreps_out={model.irreps_out}")
    print(f"  out_dim={CONFIG.model.out_dim}  params={n_params:,}")

    # One fake molecule: 6 atoms spread in 3D (well within the 5 Å cutoff).
    n = 6
    x = torch.randn(n, NODE_FEATURE_DIM)
    pos = torch.randn(n, 3) * 1.5
    batch = torch.zeros(n, dtype=torch.long)

    model.eval()
    with torch.no_grad():
        out = model(x, pos, batch=batch)
    assert out.shape == (1, CONFIG.model.out_dim), f"bad output shape: {out.shape}"
    print(f"  forward OK: {n} atoms / 1 graph -> output {tuple(out.shape)}")

    # ---- SE(3)-INVARIANCE of the prediction ------------------------------ #
    # Transform every coordinate by a random rotation R and a translation t.
    # Output irreps are scalars (0e) and edge vectors are translation-invariant
    # differences, so the prediction must be numerically unchanged.
    R = _random_rotation(seed=1)
    t = torch.tensor([2.0, -3.0, 0.5])
    pos_transformed = pos @ R.T + t            # x -> R x + t  (per row)

    with torch.no_grad():
        out_transformed = model(x, pos_transformed, batch=batch)

    max_diff = (out - out_transformed).abs().max().item()
    print(f"  SE(3) invariance: max|f(x) - f(Rx + t)| = {max_diff:.2e}")
    assert max_diff < 1e-4, "prediction is NOT invariant under rotation+translation!"

    # Two graphs in one batch should pool to [2, out_dim].
    x2 = torch.randn(n, NODE_FEATURE_DIM)
    pos2 = torch.randn(n, 3) * 1.5
    x_cat = torch.cat([x, x2], dim=0)
    pos_cat = torch.cat([pos, pos2], dim=0)
    batch_cat = torch.cat([torch.zeros(n, dtype=torch.long), torch.ones(n, dtype=torch.long)])
    with torch.no_grad():
        out_batched = model(x_cat, pos_cat, batch=batch_cat)
    assert out_batched.shape == (2, CONFIG.model.out_dim), f"bad batched shape: {out_batched.shape}"
    print(f"  batched OK: 2 graphs -> output {tuple(out_batched.shape)}")

    print("\nse3_transformer self-check passed (forward shapes + SE(3) invariance verified).")


### `src/train/train.py`

In [ ]:
%%writefile src/train/train.py
"""Train ONE model (Vanilla GNN, EGNN, or SE(3)-Transformer) on QM9.

Task 3.1: given a model name, run the training loop, log MAE/RMSE per epoch
to a CSV, and save the best-val-MAE weights. Reused as-is by
``notebooks/train_colab.ipynb`` (task 3.2) on Colab's free GPU — locally this
should only ever be smoke-tested with a tiny ``--epochs``/``--subset-size``,
never run to completion (see CLAUDE.md Phase 3 stop rule).

Usage:

    python -m src.train.train --model egnn
    python -m src.train.train --model vanilla --epochs 1 --subset-size 64  # smoke test
"""

from __future__ import annotations

import argparse
import csv
import os
import sys
import time

import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader

_REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

from src.train.config import CONFIG, set_seed  # noqa: E402
from src.data.qm9_loader import get_qm9_splits, TargetNormalizer, select_targets  # noqa: E402
from src.models.vanilla_gnn import build_vanilla_gnn  # noqa: E402
from src.models.egnn import build_egnn  # noqa: E402
from src.models.se3_transformer import build_se3_transformer  # noqa: E402

MODEL_BUILDERS = {
    "vanilla": build_vanilla_gnn,
    "egnn": build_egnn,
    "se3": build_se3_transformer,
}


def forward_model(model_name: str, model: torch.nn.Module, batch) -> torch.Tensor:
    """Dispatch to the right forward signature (each model takes different args)."""
    if model_name == "vanilla":
        return model(batch.x, batch.edge_index, batch.batch)
    # egnn and se3 both use pos; se3 ignores edge_index/edge_attr internally.
    return model(batch.x, batch.pos, batch.edge_index, batch.edge_attr, batch.batch)


def run_epoch(
    model_name: str,
    model: torch.nn.Module,
    loader: DataLoader,
    normalizer: TargetNormalizer,
    device: str,
    optimizer: torch.optim.Optimizer | None,
    max_steps: int | None = None,
) -> tuple[float, float]:
    """One pass over ``loader``. Trains if ``optimizer`` is given, else evaluates.

    Returns (MAE, RMSE) in physical units (denormalized), averaged over samples.
    """
    is_train = optimizer is not None
    model.train(is_train)

    abs_err_sum = torch.zeros(CONFIG.model.out_dim)
    sq_err_sum = torch.zeros(CONFIG.model.out_dim)
    n_samples = 0

    for step, batch in enumerate(loader):
        if max_steps is not None and step >= max_steps:
            break
        batch = batch.to(device)
        target = normalizer.normalize(select_targets(batch.y))

        with torch.set_grad_enabled(is_train):
            pred = forward_model(model_name, model, batch)
            loss = F.mse_loss(pred, target)

        if is_train:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG.train.grad_clip)
            optimizer.step()

        with torch.no_grad():
            pred_phys = normalizer.denormalize(pred)
            target_phys = normalizer.denormalize(target)
            err = pred_phys - target_phys
            abs_err_sum += err.abs().sum(dim=0).cpu()
            sq_err_sum += (err ** 2).sum(dim=0).cpu()
            n_samples += err.size(0)

    mae = (abs_err_sum / n_samples).mean().item()
    rmse = (sq_err_sum / n_samples).sqrt().mean().item()
    return mae, rmse


def train(
    model_name: str,
    epochs: int | None = None,
    subset_size: int | None = None,
    max_steps: int | None = None,
) -> str:
    """Train ``model_name`` and return the path of the saved best-weights file."""
    if model_name not in MODEL_BUILDERS:
        raise ValueError(f"model must be one of {list(MODEL_BUILDERS)}, got {model_name!r}")

    set_seed(CONFIG.seed)
    device = CONFIG.device
    epochs = CONFIG.train.epochs if epochs is None else epochs

    splits = get_qm9_splits(subset_size=subset_size)
    normalizer = TargetNormalizer.from_dataset(splits.train)
    # Move normalizer stats to the training device -- batches are moved to
    # `device` below, and normalize()/denormalize() combine them elementwise.
    normalizer = TargetNormalizer(
        mean=normalizer.mean.to(device),
        std=normalizer.std.to(device),
        targets=normalizer.targets,
    )

    train_loader = DataLoader(splits.train, batch_size=CONFIG.train.batch_size, shuffle=True)
    val_loader = DataLoader(splits.val, batch_size=CONFIG.train.batch_size, shuffle=False)

    model = MODEL_BUILDERS[model_name]().to(device)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=CONFIG.train.lr, weight_decay=CONFIG.train.weight_decay
    )

    os.makedirs(CONFIG.train.weights_dir, exist_ok=True)
    os.makedirs(CONFIG.train.log_dir, exist_ok=True)
    weights_path = os.path.join(CONFIG.train.weights_dir, f"{model_name}_best.pt")
    log_path = os.path.join(CONFIG.train.log_dir, f"{model_name}_train_log.csv")

    best_val_mae = float("inf")
    with open(log_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_mae", "train_rmse", "val_mae", "val_rmse", "seconds"])

        for epoch in range(1, epochs + 1):
            t0 = time.time()
            train_mae, train_rmse = run_epoch(
                model_name, model, train_loader, normalizer, device, optimizer, max_steps
            )
            val_mae, val_rmse = run_epoch(
                model_name, model, val_loader, normalizer, device, None, max_steps
            )
            elapsed = time.time() - t0

            writer.writerow([epoch, train_mae, train_rmse, val_mae, val_rmse, elapsed])
            f.flush()
            print(
                f"[{model_name}] epoch {epoch}/{epochs}  "
                f"train_mae={train_mae:.4f} train_rmse={train_rmse:.4f}  "
                f"val_mae={val_mae:.4f} val_rmse={val_rmse:.4f}  ({elapsed:.1f}s)"
            )

            if val_mae < best_val_mae:
                best_val_mae = val_mae
                torch.save(
                    {
                        "model_name": model_name,
                        "model_state": model.state_dict(),
                        "epoch": epoch,
                        "val_mae": val_mae,
                        "normalizer_mean": normalizer.mean,
                        "normalizer_std": normalizer.std,
                        "targets": normalizer.targets,
                    },
                    weights_path,
                )

    return weights_path


def main() -> None:
    parser = argparse.ArgumentParser(description="Train one EquiDrug model on QM9.")
    parser.add_argument("--model", required=True, choices=list(MODEL_BUILDERS))
    parser.add_argument("--epochs", type=int, default=None)
    parser.add_argument("--subset-size", type=int, default=None)
    parser.add_argument(
        "--max-steps",
        type=int,
        default=None,
        help="Cap batches per epoch (smoke testing only; omit for real training).",
    )
    args = parser.parse_args()

    weights_path = train(
        args.model,
        epochs=args.epochs,
        subset_size=args.subset_size,
        max_steps=args.max_steps,
    )
    print(f"\nBest weights saved to: {weights_path}")


if __name__ == "__main__":
    main()


## 3. Sanity-check the written tree

In [ ]:
!find . -path ./data -prune -o -type f -name "*.py" -print -o -name "*.txt" -print | sort

## 4. Install pinned free/open-source dependencies

In [ ]:
!pip install -q -r requirements.txt
!python verify_env.py

## 5. Train all three models

Reuses the embedded `src/train/train.py` exactly as written for task 3.1 - no
Colab-specific training code. `CONFIG.device` auto-detects `cuda` here, so this
is real full-length training (unlike the local 1-step CPU smoke test). QM9
auto-downloads into `data/QM9` on first run.

In [ ]:
MODELS = ["vanilla", "egnn", "se3"]

for name in MODELS:
    print("=== Training", name, "===")
    !python -m src.train.train --model {name}

## 6. Upload weights + logs to Hugging Face Hub (free, public repo)

Create a free HF account + a **public** model repo. The login cell prefers a
Colab secret named `HF_TOKEN` (key icon in the left sidebar) so a full "Run all"
never blocks; if none is set it falls back to the interactive widget. Then set
`HF_REPO_ID` to your repo and run the upload.

In [ ]:
!pip install -q huggingface_hub

import os
from huggingface_hub import login

# Prefer a Colab secret named HF_TOKEN so 'Run all' never blocks; fall back to
# the interactive widget. The token is never hardcoded in this notebook.
token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")

if token:
    login(token=token)
    print("Logged in to Hugging Face via stored token.")
else:
    print("No HF_TOKEN secret found - opening interactive login widget.")
    login()

In [ ]:
HF_REPO_ID = "your-username/equidrug-weights"  # <-- set this to YOUR public HF model repo

from huggingface_hub import HfApi

if HF_REPO_ID.startswith("your-username/"):
    print("Skipping upload: set HF_REPO_ID to your own repo id "
          "(e.g. 'alice/equidrug-weights') and re-run this cell.")
else:
    api = HfApi()
    api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True, private=False)
    api.upload_folder(
        repo_id=HF_REPO_ID,
        repo_type="model",
        folder_path="weights",
        path_in_repo="weights",
        allow_patterns=["*.pt", "*.csv"],
    )
    print("Uploaded weights + logs to https://huggingface.co/" + HF_REPO_ID)

## 7. Confirm the upload is retrievable for free
Lists the files now on the Hub so we can confirm tasks 3.3-3.6 before closing the Colab session.

In [ ]:
from huggingface_hub import list_repo_files

if HF_REPO_ID.startswith("your-username/"):
    print("Set HF_REPO_ID above first, then re-run the upload and this cell.")
else:
    for f in list_repo_files(HF_REPO_ID):
        print(f)